## Notebook 02a —  RGB Tiling (512 × 512, stride 256)

Tiles all three streams to **512 × 512 px** with **256 px stride** (50% overlap).

```
Tiles/
  rgb/images/       <- 3-band RGB tiles
  labels/           <-  label tiles

```



## 0. Imports and paths

In [1]:
from pathlib import Path
import numpy as np
import rasterio
from rasterio.windows import Window
import pandas as pd

BASE_DIR   = Path(r"E:\THESIS_COLLINS HLORDZIE\02_PROCESSED")
SA_ROOT    = BASE_DIR
LABEL_DIR  = BASE_DIR / "Labels"
OUTPUT_DIR = BASE_DIR / "Tiles"

RGB_DIR  = OUTPUT_DIR / "rgb"    / "images"
LBL_OUT  = OUTPUT_DIR / "labels"

for d in [RGB_DIR, LBL_OUT]:
    d.mkdir(parents=True, exist_ok=True)

TILE_SIZE  = 512
STRIDE     = 256
BACKGROUND = 255
SKIP_SAS   = {"SA012_10445982"}

print(f"Tile size : {TILE_SIZE}px")
print(f"Stride    : {STRIDE}px")
print("Config ready.")

Tile size : 512px
Stride    : 256px
Config ready.


## 1. Helper functions

In [2]:
def get_sa_id(folder_name):
    return folder_name.split("_", 1)[-1]

def find_file(directory, keyword):
    for f in directory.iterdir():
        if keyword in f.name and f.suffix in (".tif", ".tiff"):
            return f
    return None

def pad_array(arr, target_h, target_w, pad_value=0):
    _, h, w = arr.shape
    if h == target_h and w == target_w:
        return arr
    out = np.full((arr.shape[0], target_h, target_w),
                  fill_value=pad_value, dtype=arr.dtype)
    out[:, :h, :w] = arr
    return out

def read_window(src, row_off, col_off, tile_h, tile_w, pad_value=0):
    h, w   = src.height, src.width
    read_h = min(tile_h, h - row_off)
    read_w = min(tile_w, w - col_off)
    data   = src.read(window=Window(col_off, row_off, read_w, read_h))
    return pad_array(data, tile_h, tile_w, pad_value=pad_value)

def tile_transform(base_tf, col_off, row_off):
    x = base_tf.c + col_off * base_tf.a
    y = base_tf.f + row_off * base_tf.e
    return rasterio.Affine(base_tf.a, 0.0, x, 0.0, base_tf.e, y)

def save_tile(arr, path, transform, crs, dtype="uint8", nodata=None):
    with rasterio.open(
        path, "w", driver="GTiff", dtype=dtype,
        width=arr.shape[2], height=arr.shape[1], count=arr.shape[0],
        crs=crs, transform=transform, compress="lzw", nodata=nodata,
    ) as dst:
        dst.write(arr)

print("Helpers defined.")

Helpers defined.


## 2. Estimate tile count before running

In [3]:
sa_folders = sorted([
    d for d in SA_ROOT.iterdir()
    if d.is_dir() and d.name.startswith("SA") and d.name not in SKIP_SAS
])

total_est = 0
print(f"{'SA':30s} {'H':>6} {'W':>6} {'Est tiles':>10}")
print("-" * 55)
for sa_dir in sa_folders:
    sa_id      = get_sa_id(sa_dir.name)
    label_path = LABEL_DIR / f"{sa_id}_labels.tif"
    if not label_path.exists():
        continue
    with rasterio.open(label_path) as src:
        h, w = src.height, src.width
    n = len(range(0, h, STRIDE)) * len(range(0, w, STRIDE))
    total_est += n
    print(f"  {sa_dir.name:30s} {h:>6} {w:>6} {n:>10,}")
print("-" * 55)
print(f"  {'TOTAL (before filtering)':30s} {'':>6} {'':>6} {total_est:>10,}")

SA                                  H      W  Est tiles
-------------------------------------------------------
  SA010_10305790                  10900  11125      1,892
  SA011_10445787                  11490  11186      1,980
  SA013_10565691                  11460  11517      2,025
  SA014_10595716                  11591  11473      2,070
  SA015_10605304                  11930  11585      2,162
  SA016_10765685                  11750  11810      2,162
  SA017_10865406                  11420  11173      1,980
  SA018_10865805                  11263  11772      2,024
  SA019_11125706                  12155  11678      2,208
  SA01_9545448                    11873  11972      2,209
  SA020_11226291                  11083  11249      1,936
  SA02_9805753                    11559  11289      2,070
  SA03_9965805                    11631  11350      2,070
  SA04_10085703                   11395  11693      2,070
  SA05_10125706                   11509  11248      1,980
  SA06_10145814   

## 3. Delete existing tiles and run tiling

In [4]:
import shutil

# Clear existing tiles
for d in [RGB_DIR, LBL_OUT]:
    for f in d.glob("*.tif"):
        f.unlink()
    # Also clear split subfolders if they exist
    for split in ["train", "val", "test"]:
        split_dir = d / split
        if split_dir.exists():
            shutil.rmtree(split_dir)

print("Existing tiles cleared. Starting tiling ...", flush=True)

summary = []

for sa_dir in sa_folders:
    sa_name    = sa_dir.name
    sa_id      = get_sa_id(sa_name)
    rgb_path   = find_file(sa_dir, f"{sa_id}_stack_RGB")
    label_path = LABEL_DIR / f"{sa_id}_labels.tif"

    missing = []
    if rgb_path is None:       missing.append("RGB")
    if not label_path.exists(): missing.append("labels")
    if missing:
        print(f"  [SKIP] {sa_name} — missing: {missing}", flush=True)
        summary.append({"SA": sa_name, "tiles_saved": 0, "status": f"missing {missing}"})
        continue

    saved = 0

    with (rasterio.open(rgb_path)   as rgb_src,
          rasterio.open(label_path) as lbl_src):

        height, width = lbl_src.height, lbl_src.width
        crs       = lbl_src.crs
        transform = lbl_src.transform

        for r_idx, row in enumerate(range(0, height, STRIDE)):
            for c_idx, col in enumerate(range(0, width, STRIDE)):

                tile_name = f"{sa_id}_r{r_idx:04d}_c{c_idx:04d}.tif"
                tf        = tile_transform(transform, col, row)

                lbl_tile = read_window(lbl_src, row, col,
                                       TILE_SIZE, TILE_SIZE,
                                       pad_value=BACKGROUND)
                if np.all(lbl_tile == BACKGROUND):
                    continue

                rgb_tile = read_window(rgb_src, row, col,
                                       TILE_SIZE, TILE_SIZE, pad_value=0)

                save_tile(rgb_tile, RGB_DIR / tile_name, tf, crs,
                          dtype=rgb_src.dtypes[0])
                save_tile(lbl_tile, LBL_OUT / tile_name, tf, crs,
                          dtype="uint8", nodata=BACKGROUND)

                saved += 1

    print(f"  {sa_name:30s}  {saved:5d} tiles", flush=True)
    summary.append({"SA": sa_name, "tiles_saved": saved, "status": "ok"})

summary_df = pd.DataFrame(summary)
total = summary_df["tiles_saved"].sum()
print(f"\nTotal tiles saved: {total:,}")
display(summary_df)

Existing tiles cleared. Starting tiling ...
  SA010_10305790                   1856 tiles
  SA011_10445787                    688 tiles
  SA013_10565691                   2009 tiles
  SA014_10595716                   2008 tiles
  SA015_10605304                   2024 tiles
  SA016_10765685                   1237 tiles
  SA017_10865406                   1298 tiles
  SA018_10865805                   1165 tiles
  SA019_11125706                    836 tiles
  SA01_9545448                     1889 tiles
  SA020_11226291                     21 tiles
  SA02_9805753                      364 tiles
  SA03_9965805                      153 tiles
  SA04_10085703                     904 tiles
  SA05_10125706                    1887 tiles
  SA06_10145814                    1955 tiles
  SA07_10165835                     424 tiles
  SA08_10165859                     682 tiles
  SA09_10185850                     791 tiles

Total tiles saved: 22,191


,SA,tiles_saved,status
0,SA010_10305790,1856,ok
1,SA011_10445787,688,ok
2,SA013_10565691,2009,ok
3,SA014_10595716,2008,ok
4,SA015_10605304,2024,ok
5,SA016_10765685,1237,ok
6,SA017_10865406,1298,ok
7,SA018_10865805,1165,ok
8,SA019_11125706,836,ok
9,SA01_9545448,1889,ok


## 4. Verify tile counts match

In [5]:
rgb_tiles = list(RGB_DIR.glob("*.tif"))
lbl_tiles = list(LBL_OUT.glob("*.tif"))

print(f"RGB    tiles : {len(rgb_tiles):,}")
print(f"Label  tiles : {len(lbl_tiles):,}")

if len(rgb_tiles) == len(lbl_tiles):
    print("\nAll streams match. Ready for splitting.")
else:
    print("\nWARNING — counts do not match. Investigate before splitting.")

RGB    tiles : 22,191
Label  tiles : 22,191

All streams match. Ready for splitting.
